In [ ]:
import warnings,re,copy,numpy as np,seaborn as s,pandas as pd,matplotlib.pyplot as plt; from datetime import datetime; from tqdm import tqdm; import matplotlib.font_manager as f
f.fontManager.addfont("s.ttf");plt.rcParams['axes.unicode_minus']=0;plt.rcParams['font.sans-serif']=[f.FontProperties(fname="s.ttf").get_name()];rd=round;r=range; 测挂=0; 高挂=9
左=lambda 价:np.insert(价[:-1],0,价[0]); 右=lambda 价:np.insert(价[1:],-1,价[-1]); pp=lambda a:np.array(a); 变幅=lambda 次,首:(次-首)/首*100; 掩点=lambda 掩:np.where(掩)[0]; 串否=0 
抠数=lambda l,位:pp([int(str(日).split('-')[位]) for 日 in l]); 热图=lambda x,矩,c=0:s.heatmap(矩,cmap='viridis',yticklabels=r(-10,11),xticklabels=r(-10,11),center=c,annot=矩,ax=x) 
标=[['价','高','低','开','前'],['开收','中收','开中']]; 颜=['b','r','g','orange','purple']; warnings.filterwarnings('ignore'); 集=['今全价.csv','去全价.csv','前全价.csv']
# 文集=['d201.csv','d223.csv','d245.csv']; 额列 = [pd.read_csv(f,header=None).iloc[:,3:] for f in 文集] #---------------若已下多个年份文件,用这两行进行合并------------
# 基列=pd.read_csv(文集[0],header=None).iloc[:,:3]; pd.concat([基列]+额列,axis=1).to_csv('d205.csv',index=False,header=False) 
def 画图(x,矩):刻=r(-10,11);[x.plot(刻,值[1:22],c=颜[i],label=标[len(矩)<4][i])for i,值 in enumerate(矩)];x.set_xticks(刻);x.grid(True);x.legend();np.nanmin(矩)<0 and x.set_ylim(-1,1)
def 解构(益矩,长矩): return (np.nanmean(np.where(益矩!=0,益矩/长矩,np.nan),axis=2)),(np.sum(np.where(长矩>0,1,0),axis=2)) # 时间=时间[细:=抠数(时间,0)*年号==年号*年号];df=df[:,细] 
def 当星反应(周号,名): # 文件均含开/高/收/低四个价格,停牌时股价可能为0,这时换成nan防画入||益矩:[因子数,点数,天数],方矩则存储对应例数 # 48-6160 # 88-8927 # 57-6668 # 72-8003 .iloc[:,:600]
    起=3; f=pp(l:=pd.read_csv(名,encoding=['utf-8','gbk'][0]).replace(0,np.nan)); 掩=pp([False]*(f.shape[1]-起)); 掩[-[1000,2400][名=='今全价.csv']:]=True; df=f[:,起:][:,掩].copy() 
    价数=[4,5][f[3,1]==f[4,1]]; 时间=pp(l.columns[起:][掩]); 长=len(时间); 去端=pp([True]*长); 去端[0]=去端[-1:]=False; 累矩=np.zeros([21,21,长]); 长矩=累矩.copy(); 宽=11; 低挂=0 
    周集=[datetime.strptime(日,'%Y-%m-%d').weekday()+1 for 日 in 时间]; print(周集[:5],时间[:5],长); 益矩=np.zeros([[5,3][买点==2],23,长]); 方矩=益矩.copy(); 量矩=[]; 宽矩=np.zeros(长) 
    置矩=np.zeros([21,21,长]); 数矩=置矩.copy(); 邻变=lambda a:[变幅(a[x+1],a[x]) for x in r(len(a)-1)]; 邻变=lambda a:[变幅(x,a[0]) for x in a[1:]] 
    for 股号 in tqdm(r(0,min(len(f)//价数,20000)),position=0,leave=True): # 0.开盘买次日卖|1.收盘买次日卖|2.开盘买当日收盘卖(买卖点其一也可改成中盘)||涨表和左涨表是收/开盘买的自变量 
        线表=[df[i::价数][股号] for i in r(价数)]; 线表=(*线表,线表[0]) if 价数==4 else 线表; 价线,高线,低线,开线,中线=线表 # 无中价则由开价替||一般5因变量,当日买卖只考虑买点点位 
        涨表=[变幅(l,左(价线)) for l in 线表]; 价涨,高涨,低涨,开涨,中涨=涨表; 涨表[-1]=前涨=左(价涨); 左涨表=[左(l) for l in 涨表]; 左涨表[-1]=开涨 # 中涨更为前涨,(开买用的)左前涨更为开涨  
        生涨=lambda 起线:[变幅(右(l),起线) for l in [开线,价线,中线]]; 开开涨,开价涨,开中涨=开表=生涨(开线); 价开涨,价价涨,价中涨=生涨(价线) # 2买点*3卖点得6因变量(中线因缺自变量而不作买点) 
        开当涨,开高涨,开低涨,_,_=[变幅(l,开线) for l in 线表]; 右开涨=o=右(开涨); 右高涨=h=右(高涨); 开半涨=变幅(中线,开线); 中当涨=变幅(价线,中线); 左后涨=左(中当涨); 左中涨=左(中涨) 
        开价涨=np.where(开挂<o,开开涨,np.where(高挂<h,开当涨+高挂,开价涨)); 价价涨=np.where(开挂<o,o,np.where(高挂<h,高挂,价价涨)); 左价涨,左高涨,左低涨,左开涨,_=左涨表; 左前涨=左(前涨) 
        开中涨=np.where(开挂<o,开开涨,开中涨); 挂涨=lambda l:np.where(低挂>开涨,l,np.where(低挂>低涨,l+开涨-低挂,0)); 挂开涨,挂价涨,挂中涨=[挂涨(l) for l in 开表] # 全天逢低买入的跨日收益
        右价涨=右(价涨); 双价涨=变幅(右(价线),左(价线))  
        if 买点==1: 涨表=[价涨-中涨,高涨-价涨,低涨-价涨,高涨-开涨,低涨-开涨] # 这里涨表原为价涨,高涨,低涨,开涨,中涨,是隔日策略的基准列表,故记得标注以免影响可视化------
        for 点 in 掩点((价涨<-9)*(价涨==低涨)*去端): # 跌幅大于9且等于最低价才判跌停,此跌停是对卖日而言,'×当涨'的买卖点均当天,卖点改迁即可,'开×涨'/'价×涨'买点为前天,与涨矢的索引均需减一
            截=lambda a:min(a,长-2); 允跌=8; 可=lambda 量:next((点+i for i in r(允跌+1) if (量[截(点+i)]>-9 or 量[截(点+i)]!=低涨[截(点+i)])),截(点+允跌)) 
            组=[开线[可(开涨)],价线[可(价涨)],中线[可(中涨)]]; 价卖=组[1]; 开当涨[点]=变幅(价卖,开线[点]); 中当涨[点]=变幅(价卖,中线[点]); 开半涨[点]=变幅(组[-1],开线[点])
            q=点-1; 开开涨[q],开价涨[q],开中涨[q]=[变幅(l,开线[q]) for l in 组]; 价开涨[q],价价涨[q],价中涨[q]=[变幅(l,价线[q]) for l in 组] 
        for 点 in 掩点((pp(周集)*周号==周号*周号)*去端): # 点即买点,三条件:1.当日收盘未涨停而可买入|2.当日属特定星期|3.排除无参考和无验证的端部||下面对益限幅以过滤因停牌产生的跳空点 
            if 买点==3 and len(价线)-点>23 and abs(价涨[点])<10.5 and abs(前涨[点])<22 and abs(左前涨[点])<22 and 高涨[点]>-19 and not (价涨[点]==0 and 价涨[点-1]==0) and rd(价涨[点])==0 and rd(前涨[点])==0 and rd(左前涨[点])==0 and abs(rd(右价涨[点]))<11: # 三连横盘接一日随变后的趋势可视化
                置矩[i:=rd(右价涨[点])+10,:,点]+=pp(邻变(价线[点:点+22])); 数矩[i,:,点]+=1; continue 
            if 买点==4 and 开涨[点]<9 and all([abs(x[点])<10.4 for x in 左涨表]) and 开中涨[点]<25: # 这是隔夜单所挂点数可变的跨日策略
                for 挂点 in r(-7,7): 累矩[i:=10-挂点,rd(前涨[点])+10,点]+=np.where(挂点/4>低涨[点],双价涨[点]-min(挂点/4,开涨[点]),0); 长矩[i,rd(前涨[点])+10,点]+=1 
            if 买点==0 and 开涨[点]<9 and all([abs(x[点])<10.4 for x in 左涨表]) and 开中涨[点]<25: # 0.开盘买入,次日开/中/收卖出(原标况是高9低5价1,现为开涨过1并限高低,暂不需累矩)
                # if not 左开涨[点]>1.5 or not 左低涨[点]>-2 or not 左高涨[点]<9.5: continue # -------------------------------------------------------------------  
                if not 左价涨[点]<9.5 or not 左高涨[点]>9.5 or not abs(左低涨[点])<2 or not abs(左开涨[点])<1: continue # ---------------------------------------
                for i in r(len(益矩)): 益矩[i,j:=rd(左涨表[i][点])+宽,点]+=开价涨[点]; 方矩[i,j,点]+=1 # 据卖点选: 开开涨/开价涨/开中涨, 或者: 挂开涨/挂价涨/挂中涨
            if 买点==1 and 价涨[点]<9 and all([abs(x[点])<10.4 for x in 涨表]) and abs(中涨[点])<10.4: # 1.收盘买入,次日开/中/收卖出(标况要求收盘价贴合最低价,前涨正负3高涨正3,前消则微逊) 
                # if np.isnan(高涨[点]) or not 价涨[点]-低涨[点]<0.5 or not abs(前涨[点])<3 or not 0<高涨[点]<3 or not 开涨[点]>-1: continue # --------------------
                # if np.isnan(高涨[点]) or not 高涨[点]-价涨[点]<3 or not -2<低涨[点]-价涨[点]<-1 or not 低涨[点]-开涨[点]>-2 or not 0<价涨[点]<4 or not 高涨[点]-低涨[点]>1: continue
                if np.isnan(高涨[点]) or not 高涨[点]-价涨[点]<3 or not -2<低涨[点]-价涨[点] or not -3<价涨[点]<3 or not 价涨[点]-中涨[点]<0 or not 0<中涨[点]-开涨[点]: continue 
                for i in r(len(益矩)): 益矩[i,j:=rd(涨表[i][点])+宽,点]+=价价涨[点]; 方矩[i,j,点]+=1 # 据卖点选: 价开涨/价价涨/价中涨||上行是带中值而上上行是去掉中值的平替 ↑
            if 买点==2 and all([abs(x[点])<10.4 for x in [开涨,中涨]]): # 将日内收益拆分成三段考虑:"开→收/中→收/开→中",累矩暂设为延迟追涨,其同时需前两日阴线以提高鲁棒性 
                if np.isnan(左价涨[点]) or np.isnan(开中涨[点]) or not 左价涨[点]>9.5 or not 左前涨[点]<0 or not 左(左前涨)[点]<0: continue # -----------------
                for i,(涨,当涨) in enumerate([(开涨,开当涨),(中涨,中当涨),(开涨,开半涨)]): k=rd(涨[点])+宽; 益矩[i,k,点]+=当涨[点]; 方矩[i,k,点]+=1; 目涨=[左价涨,左后涨,左开涨,左中涨][1] 
                # if 目涨[点]>-12 and abs(rd(目涨[点]))<11: 累矩[i:=rd(目涨[点])+10,j:=rd(开涨[点])+10,点]+=开当涨[点]; 长矩[i,j,点]+=1 # 如上,目涨即前日涨况,有四候选,一般是0,但12也有惊喜
                # 累矩[i:=rd(开涨[点])+10,j:=rd(中涨[点])+10,点]+=中当涨[点]; 长矩[i,j,点]+=1 # 考察后半涨态,可迁此到上两代码段中(收盘买为:价涨+前涨=价价涨)---------- 
                # for 挂 in r(1,顶:=12-rd(开涨[点])): 挂点=挂%(顶-1); 累矩[20-挂点,i:=rd(开涨[点])+10,点]+=np.where(挂<开高涨[点],挂,开当涨[点]); 长矩[20-挂点,i,点]+=1 # 卖出挂单
                for 挂 in r(1,底:=12+rd(开涨[点])): 挂点=挂%(底-1); 累矩[20-挂点,i:=rd(开涨[点])+10,点]+=np.where(-挂点>开低涨[点],开价涨[点]+挂点,0); 长矩[20-挂点,i,点]+=1 # 买入挂单--
                if rd(开涨[点]) in [-3,-2,-1]: 挂=1; 宽矩[点]+=1; 量矩.append(涨:=np.where(-挂>开低涨[点],开价涨[点]+挂,0)) # 开价涨/开当涨/开开涨 
                # (顶-1=11-点)是开盘点到最高点的距离加1,整除先转此1为0得累矩的20位即矩底,值对应原不能成交的11点故为开当涨||下面买入挂单是其镜像过程,值若对应难交点则恒0无意义,故亦转0为开当涨
                # 累矩[j:=rd(左价涨[点])+10,i:=rd(开涨[点])+10,点]+=np.where(-挂点>开低涨[点],开价涨[点]+挂点,0); 长矩[j,i,点]+=1 # 此处的挂点在函数外部就已定义
    if 买点==3: 置矩,数矩=解构(置矩,数矩); x=plt.subplots(1,2,figsize=(12,5))[1]; 热图(x[0],(置矩*10).round(0),0); 热图(x[1],(数矩/10).round(0),0); plt.show(); return 1
    if 测挂==1: print("开:",开挂,"高:",高挂,"均益:",np.nanmean(np.where(益矩[0].sum(0)!=0,益矩[0].sum(0)/方矩[0].sum(0),np.nan),axis=0).round(4)); return 0 
    if 宽矩.max()==0 and not 买点==4: 益矩,方矩=解构(益矩,方矩); x=plt.subplots(2,1,figsize=(6,4))[1]; 画图(x[0],方矩); 画图(x[1],益矩); plt.show() 
    if 宽矩.max()>0: print("益例:",pp(量矩)[-10:].round(4),"周号:",周号,"占宽:",np.sum(宽矩>0),"均益:",pp(量矩).mean().round(4),"交率:",((pp(量矩)!=0).sum()/len(量矩)).round(4)) 
    if 有累:=累矩.max()>0: 累矩,长矩=解构(累矩,长矩); x=plt.subplots(1,2,figsize=(12,5))[1]; 热图(x[0],(累矩*10).round(0),0); 热图(x[1],(长矩/10).round(0),None); plt.show() 
    if 串否==1: return [益矩,(累矩*10).round(0)][有累] # 仅买点为3且累矩有值时返回累矩
买点=0 
开挂=2 
# for i in tqdm(r(9)): 测挂=1; 高挂=i; 当星反应(0,'前全价.csv') # 测"开挂/高挂/(开盘低买时的)挂点",测高挂时请标注上面的设值 
当星反应(0,'dd26.csv') # 单文件 
# 当星反应(0,'价b.csv') # 单文件 
# for 周号 in r(1,6): 当星反应(周号,'今全价.csv') # 逐星期 || 下为多文件并融合,即'今全价.csv','去全价.csv','前全价.csv'
串否=1; a=np.minimum.reduce([当星反应(0,名) for 名 in 集]); x=plt.subplots(1,1,figsize=(6,[2,5][回热:=len(a)>6]))[1]; 热图(x,a) if 回热 else 画图(x,a); plt.show() 